# Project Journal 04: Direct 2010 Benchmark Audit

This entry records the strongest field-facing check in the project so far: a direct same-year comparison between a 2010 monitor prediction and the public 2010 Sleipner plume-support benchmark.


## Context

After the multi-vintage `p07` review, the project still needed to show that the field story was not just a convenient interpretation of earlier-vintage support trends.

This stage added a direct `94p10 -> 10p10` audit with three goals:

- test the constrained hybrid against a same-year 2010 structural benchmark
- compare it to the stronger cross-equalized baseline under the same benchmark geometry
- keep the run provenance explicit by reusing the already-trained synthetic artifacts through a dedicated `artifacts_root`

This is the point where the project became much more suitable for a short paper claim, because the field comparison tightened around a public benchmark instead of a mostly qualitative visual story.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

ROOT = Path.cwd().resolve()
if not (ROOT / 'runs').exists():
    ROOT = ROOT.parent.resolve()

def load_json(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return json.load(handle)

def show_images(image_paths, titles, figsize=(18, 8)):
    fig, axes = plt.subplots(1, len(image_paths), figsize=figsize)
    if len(image_paths) == 1:
        axes = [axes]
    for ax, image_path, title in zip(axes, image_paths, titles):
        image = mpimg.imread(image_path)
        ax.imshow(image)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

ROOT

p07_summary = load_json(ROOT / 'runs' / 'sleipner_manifest' / 'results' / 'summary.json')
p10_summary = load_json(ROOT / 'runs' / 'sleipner_manifest_p10' / 'results' / 'summary.json')
p10_field = p10_summary['Field']
p10_overview = p10_summary['Overview']
p10_pair = p10_field['pairs']['sleipner_2010_inline_1840_p10_mid']
p10_field


## What Changed This Stage

The main additions in this stage were:

- direct export of the `94p10` baseline and `10p10` monitor on the same inline
- benchmark-derived storage mask and 2010 plume-support envelope on the `p10` geometry
- a dedicated `p10` config that saves outputs separately while reusing the trained model artifacts from the main field prototype
- an explicit benchmark note that the 2010 support product is a structural envelope, not an exact pixel-level plume label

This stage was designed to answer the question: does the project still look useful when the benchmark comparison becomes stricter?


In [ ]:
summary_rows = [
    {
        'case': 'p07 constrained 2001',
        'trace_iou_2010_support': p07_summary['Field']['pairs']['sleipner_2001_inline_1840_mid']['hybrid_ml_constrained']['trace_iou_with_2010_support'],
    },
    {
        'case': 'p07 constrained 2004',
        'trace_iou_2010_support': p07_summary['Field']['pairs']['sleipner_2004_inline_1840_mid']['hybrid_ml_constrained']['trace_iou_with_2010_support'],
    },
    {
        'case': 'p07 constrained 2006',
        'trace_iou_2010_support': p07_summary['Field']['pairs']['sleipner_2006_inline_1840_mid']['hybrid_ml_constrained']['trace_iou_with_2010_support'],
    },
    {
        'case': 'p10 cross-equalized 2010',
        'trace_iou_2010_support': p10_pair['cross_equalized_difference']['trace_iou_with_2010_support'],
    },
    {
        'case': 'p10 constrained hybrid 2010',
        'trace_iou_2010_support': p10_pair['hybrid_ml_constrained']['trace_iou_with_2010_support'],
    },
]

display(pd.DataFrame(summary_rows))

p10_method_rows = [
    {
        'method': 'cross_equalized_difference',
        'predicted_trace_fraction': p10_pair['cross_equalized_difference']['predicted_trace_fraction'],
        'trace_iou_with_2010_support': p10_pair['cross_equalized_difference']['trace_iou_with_2010_support'],
        'trace_fraction_outside_2010_support': p10_pair['cross_equalized_difference']['trace_fraction_outside_2010_support'],
    },
    {
        'method': 'hybrid_ml_constrained',
        'predicted_trace_fraction': p10_pair['hybrid_ml_constrained']['predicted_trace_fraction'],
        'trace_iou_with_2010_support': p10_pair['hybrid_ml_constrained']['trace_iou_with_2010_support'],
        'trace_fraction_outside_2010_support': p10_pair['hybrid_ml_constrained']['trace_fraction_outside_2010_support'],
    },
]

display(pd.DataFrame(p10_method_rows))

raw_field_rows = [
    {
        'method': 'plain_ml',
        'predicted_fraction': p10_pair['plain_ml']['predicted_fraction'],
        'outside_reservoir_fraction': p10_pair['plain_ml']['outside_reservoir_fraction'],
    },
    {
        'method': 'hybrid_ml',
        'predicted_fraction': p10_pair['hybrid_ml']['predicted_fraction'],
        'outside_reservoir_fraction': p10_pair['hybrid_ml']['outside_reservoir_fraction'],
    },
]

display(pd.DataFrame(raw_field_rows))


## Current Results

This stage gave the strongest field-facing result in the project so far.

What the direct 2010 audit supports:

- the constrained hybrid reaches very strong overlap with the 2010 support benchmark
- the constrained hybrid is far more selective than the cross-equalized baseline on the same geometry
- the `p10` result is consistent with the earlier `p07` trend that the constrained support mapping becomes more benchmark-aligned at later times

What still needs to be said honestly:

- the raw hybrid output is still not field-ready without structural constraining
- the benchmark support comparison is still about lateral support, not exact pixel-level plume boundaries
- the zero outside-reservoir score for the constrained output is achieved after applying the benchmark storage mask

That makes the paper claim narrower, but also much safer: the method is useful as a benchmark-constrained support-mapping workflow, not yet as a fully unconstrained field inversion result.


In [ ]:
show_images(
    [
        ROOT / 'runs' / 'sleipner_manifest_p10' / 'results' / 'figures' / 'field_sleipner_2010_inline_1840_p10_mid_cross_equalized_difference.png',
        ROOT / 'runs' / 'sleipner_manifest_p10' / 'results' / 'figures' / 'field_sleipner_2010_inline_1840_p10_mid_hybrid_example.png',
        ROOT / 'runs' / 'sleipner_manifest_p10' / 'results' / 'figures' / 'field_sleipner_2010_inline_1840_p10_mid_hybrid_constrained.png',
    ],
    ['2010 cross-equalized baseline', '2010 raw hybrid', '2010 constrained hybrid'],
    figsize=(18, 6),
)


## Open Problems

Even after the direct 2010 benchmark audit, a few limits remain important:

- the benchmark support envelope is not a dense plume label, so paper language should stay focused on support overlap and selectivity
- the unconstrained model still performs poorly on field leakage metrics
- the current strongest claim depends on benchmark-constrained postprocessing and shared thresholds, which should be ablated explicitly in the paper

In other words, the project now has a real field-backed claim, but it is still a carefully bounded claim.


In [ ]:
print('Run provenance')
display(pd.DataFrame([p10_overview]))
print()
print(p10_field['support_note'])
print()
print('Constraint metadata for the constrained 2010 result:')
display(pd.DataFrame([p10_pair['hybrid_ml_constrained']['constraint_metadata']]))


## Next Stage

The next useful notebook should probably be a paper-evidence notebook rather than another development journal entry.

The minimum defensible follow-up would be:

1. one table that compares synthetic, OOD, `p07`, and direct `p10` field evidence
2. one figure panel for the `2010` cross-equalized vs raw hybrid vs constrained hybrid comparison
3. one ablation that removes either shared thresholds or the benchmark reservoir mask to show why the constrained protocol matters

Series note:

- This entry is the current strongest field audit in the journal.
- A future `05_...` notebook should focus on paper figures, tables, and ablations rather than more free-form experimentation.
